# 05: SHAP Explainability
Explain Random Forest predictions per technique

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, shap, os, pickle, warnings
warnings.filterwarnings('ignore')
shap.initjs()
print('=== NOTEBOOK 05: SHAP EXPLAINABILITY ===')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()
label_map = pd.read_csv('../data/processed/label_map.csv')
from sklearn.preprocessing import LabelEncoder
for col in X_test.select_dtypes(include='object').columns:
    le = LabelEncoder(); X_test[col] = le.fit_transform(X_test[col].astype(str))
X_test = X_test.fillna(0)
actual_classes = sorted(y_test.unique())
target_names = [label_map.loc[label_map['encoded'] == i, 'technique'].values[0] for i in actual_classes]
with open('../models/random_forest.pkl', 'rb') as f: model = pickle.load(f)
print(f'Loaded RF model. Classes: {target_names}')
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
os.makedirs('../results/figures', exist_ok=True)
for i, class_name in enumerate(target_names):
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values[i], X_test, show=False, max_display=15)
    plt.title(f'SHAP: {class_name}', fontsize=14); plt.tight_layout()
    safe = class_name.replace(' ', '_').replace('/', '_')
    plt.savefig(f'../results/figures/shap_summary_{safe}.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Saved: shap_summary_{safe}.png')
for i, class_name in enumerate(target_names):
    class_idx = np.where(y_test.values == actual_classes[i])[0]
    if len(class_idx) == 0: continue
    sample_idx = class_idx[0]
    plt.figure(figsize=(16, 4))
    shap.force_plot(explainer.expected_value[i], shap_values[i][sample_idx, :], X_test.iloc[sample_idx, :], feature_names=X_test.columns.tolist(), show=False, matplotlib=True)
    safe = class_name.replace(' ', '_').replace('/', '_')
    plt.savefig(f'../results/figures/shap_force_{safe}.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Saved: shap_force_{safe}.png')
print('\n=== SHAP COMPLETE ===')
